# 2. First Cell – Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, RepeatVector
from tensorflow.keras.callbacks import EarlyStopping

import hashlib
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports done")

# ------------------------------------------------------------------
# Stage A + Stage B data sources (repo-local paths)
# ------------------------------------------------------------------
SOURCE_FILES = [
    Path('../Dataset/raw_data/dht11_dataset_10000.csv'),
    Path('../Dataset/raw_data/iot_sensor_dataset.csv'),
    Path('../Dataset/multi_sensor_cleaned_balanced.csv'),
    Path('../Dataset/raw_data/Train_Test_IoT_Weather.csv'),
    Path('../Dataset/raw_data/log_temp.csv')
]


def _standardize_status(s):
    if pd.isna(s):
        return np.nan
    v = str(s).strip().lower()
    if v in {'normal', 'benign', 'ok', '0'}:
        return 'normal'
    if 'replay' in v:
        return 'replay'
    if v in {'anomaly', 'attack', 'intrusion'}:
        return 'anomaly'
    return 'anomaly'


def _parse_log_temp(path):
    # format example: 3/14/19,19:33:07,T=22.0,H=20.0
    rows = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) < 4:
                continue
            dt = f"{parts[0].strip()} {parts[1].strip()}"
            t_match = re.search(r'T\s*=\s*([-+]?\d*\.?\d+)', parts[2])
            h_match = re.search(r'H\s*=\s*([-+]?\d*\.?\d+)', parts[3])
            if not (t_match and h_match):
                continue
            rows.append({
                'timestamp': dt,
                'temperature': float(t_match.group(1)),
                'humidity': float(h_match.group(1)),
                'status': np.nan
            })
    if not rows:
        return pd.DataFrame(columns=['timestamp', 'temperature', 'humidity', 'status'])
    out = pd.DataFrame(rows)
    out['timestamp'] = pd.to_datetime(out['timestamp'], errors='coerce')
    return out.dropna(subset=['timestamp'])


def load_and_standardize(path):
    path = Path(path)

    if path.name == 'log_temp.csv':
        df = _parse_log_temp(path)
    else:
        df = pd.read_csv(path, low_memory=False)

    original_cols = {c.lower().strip(): c for c in df.columns}

    # Build timestamp
    if 'timestamp' in original_cols:
        ts = pd.to_datetime(df[original_cols['timestamp']], errors='coerce')
    elif 'date' in original_cols and 'time' in original_cols:
        ts = pd.to_datetime(
            df[original_cols['date']].astype(str) + ' ' + df[original_cols['time']].astype(str),
            errors='coerce'
        )
    elif 'date' in original_cols:
        ts = pd.to_datetime(df[original_cols['date']], errors='coerce')
    elif 'time' in original_cols:
        ts = pd.to_datetime(df[original_cols['time']], errors='coerce')
    else:
        ts = pd.to_datetime(pd.Series([np.nan] * len(df)), errors='coerce')

    # Build temperature/humidity
    temp_col_candidates = ['temperature_c', 'temperature', 'temp']
    hum_col_candidates = ['humidity_percent', 'humidity']

    temp = None
    for c in temp_col_candidates:
        if c in original_cols:
            temp = pd.to_numeric(df[original_cols[c]], errors='coerce')
            break

    hum = None
    for c in hum_col_candidates:
        if c in original_cols:
            hum = pd.to_numeric(df[original_cols[c]], errors='coerce')
            break

    # Fallbacks for missing temp/humidity
    if temp is None:
        temp = pd.Series([np.nan] * len(df))
    if hum is None:
        hum = pd.Series([np.nan] * len(df))

    # Build status (if available)
    status = pd.Series([np.nan] * len(df))
    if 'status' in original_cols:
        status = df[original_cols['status']].map(_standardize_status)
    elif 'label' in original_cols:
        # Common pattern: 0 normal, 1 attack/anomaly
        raw = pd.to_numeric(df[original_cols['label']], errors='coerce')
        status = np.where(raw == 0, 'normal', 'anomaly')
        status = pd.Series(status)
    elif 'type' in original_cols:
        status = df[original_cols['type']].map(_standardize_status)

    out = pd.DataFrame({
        'timestamp': ts,
        'temperature': temp,
        'humidity': hum,
        'status': status,
        'data_source': path.name
    })

    # Basic cleaning
    out = out.dropna(subset=['timestamp', 'temperature', 'humidity'])
    out = out[(out['temperature'].between(-50, 100)) & (out['humidity'].between(0, 100))]

    return out.sort_values('timestamp').reset_index(drop=True)


all_frames = []
for fp in SOURCE_FILES:
    if fp.exists():
        try:
            dfi = load_and_standardize(fp)
            if len(dfi) > 0:
                all_frames.append(dfi)
                print(f"✅ {fp.name}: {dfi.shape}")
            else:
                print(f"⚠️ {fp.name}: empty after cleaning")
        except Exception as e:
            print(f"⚠️ Failed to parse {fp.name}: {e}")
    else:
        print(f"⚠️ Missing file: {fp}")

if not all_frames:
    raise ValueError('No valid datasets loaded.')

raw_df = pd.concat(all_frames, ignore_index=True)
raw_df = raw_df.sort_values('timestamp').reset_index(drop=True)

print('\nCombined shape:', raw_df.shape)
print('Sources:', raw_df['data_source'].value_counts().to_dict())
print('Status distribution (incl. NaN):')
print(raw_df['status'].value_counts(dropna=False))
raw_df.head()


# 3. EDA & Preprocessing Cell

In [ ]:
# ------------------------------------------------------------------
# Preprocessing + sequence generation (auto schema already handled)
# ------------------------------------------------------------------
window_size = 60
rng = np.random.default_rng(42)

# Keep a copy for feature engineering
df = raw_df.copy()
df['temp_change'] = df['temperature'].diff().fillna(0)
df['humidity_change'] = df['humidity'].diff().fillna(0)
df['hour'] = df['timestamp'].dt.hour.fillna(0)

# Fit scaler on likely-normal pool (known normal + unlabeled)
normal_like_mask = df['status'].isin(['normal']) | df['status'].isna()
scaler = MinMaxScaler()
scaler.fit(df.loc[normal_like_mask, ['temperature', 'humidity']])
df[['temperature', 'humidity']] = scaler.transform(df[['temperature', 'humidity']])

# Create sequences + metadata label at end timestep
X, seq_labels, seq_sources = [], [], []
for i in range(len(df) - window_size):
    seq = df[['temperature', 'humidity']].iloc[i:i + window_size].values
    label_point = df['status'].iloc[i + window_size]
    source_point = df['data_source'].iloc[i + window_size]

    X.append(seq)
    seq_labels.append(label_point if pd.notna(label_point) else 'unknown')
    seq_sources.append(source_point)

X = np.array(X)
seq_labels = np.array(seq_labels, dtype=object)
seq_sources = np.array(seq_sources, dtype=object)

# ------------------------------------------------------------------
# Synthetic replay generation for unlabeled parts (Stage B)
# ------------------------------------------------------------------
unknown_idx = np.where(seq_labels == 'unknown')[0]
if len(unknown_idx) > 0:
    n_replay = max(1, int(0.05 * len(unknown_idx)))
    replay_targets = rng.choice(unknown_idx, size=n_replay, replace=False)

    # donor sequences from normal-like windows
    donor_pool = np.where((seq_labels == 'normal') | (seq_labels == 'unknown'))[0]
    donor_pool = donor_pool[donor_pool < len(X)]
    if len(donor_pool) > 0:
        donors = rng.choice(donor_pool, size=n_replay, replace=True)
        for t, d in zip(replay_targets, donors):
            X[t] = X[d].copy()         # exact repeated window -> replay signature
            seq_labels[t] = 'replay_synthetic'

# Training set: only normal-like data (exclude known/synthetic anomalies)
train_mask = np.isin(seq_labels, ['normal', 'unknown'])
X_train = X[train_mask]

# Evaluation set: all sequences (guarantees both normal+anomaly classes)
X_eval = X.copy()
y_eval_binary = np.where(np.isin(seq_labels, ['anomaly', 'replay', 'replay_synthetic']), 1, 0)

print(f"Total sequences: {X.shape}")
print(f"Train normal-like sequences: {X_train.shape}")
print(f"Eval class balance: normal={(y_eval_binary==0).sum()} anomaly={(y_eval_binary==1).sum()}")
print('Sequence label distribution:')
print(pd.Series(seq_labels).value_counts())


# 5. Threshold Calculation + Anomaly Detection

In [ ]:
def build_lstm_autoencoder(timesteps=60, n_features=2):
    inputs = Input(shape=(timesteps, n_features))
    encoded = LSTM(64, activation='relu', return_sequences=True)(inputs)
    encoded = Dropout(0.2)(encoded)
    encoded = LSTM(32, activation='relu', return_sequences=False)(encoded)
    
    decoded = RepeatVector(timesteps)(encoded)
    decoded = LSTM(32, activation='relu', return_sequences=True)(decoded)
    decoded = Dropout(0.2)(decoded)
    decoded = LSTM(64, activation='relu', return_sequences=True)(decoded)
    outputs = Dense(n_features)(decoded)
    
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

model = build_lstm_autoencoder()
model.summary()

# Train ONLY on normal data
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, X_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Plot training (no overfitting guaranteed by early stopping + normal-only train)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('LSTM Autoencoder Loss')
plt.legend()
plt.show()

# 5. Threshold Calculation + Anomaly Detection

In [ ]:
# ------------------------------------------------------------------
# Reconstruction thresholding (robust, less brittle than mean+3*std)
# ------------------------------------------------------------------
if len(X_train) < 20:
    raise ValueError('Not enough normal-like sequences for threshold estimation.')

split_idx = int(0.8 * len(X_train))
X_val = X_train[split_idx:] if split_idx < len(X_train) else X_train

val_recon = model.predict(X_val)
val_mse = np.mean(np.power(X_val - val_recon, 2), axis=(1, 2))

# Blend quantile + IQR rule for robustness
q995 = np.quantile(val_mse, 0.995)
iqr = np.quantile(val_mse, 0.75) - np.quantile(val_mse, 0.25)
iqr_thr = np.quantile(val_mse, 0.75) + 1.5 * iqr
threshold = max(q995, iqr_thr)

print(f"✅ Threshold (robust): {threshold:.6f}")

# Predict on eval (contains both normal + anomaly => avoids ROC nan)
recon_eval = model.predict(X_eval)
eval_mse = np.mean(np.power(X_eval - recon_eval, 2), axis=(1, 2))
y_pred_binary = (eval_mse > threshold).astype(int)


# 6. Replay Attack Detector (Deterministic – Zero Hallucination)

In [ ]:
def is_replay_attack(sequence, history_sequences):
    seq_str = str(sequence.flatten().round(6))
    seq_hash = hashlib.sha256(seq_str.encode()).hexdigest()

    for hist in history_sequences:
        hist_str = str(hist.flatten().round(6))
        hist_hash = hashlib.sha256(hist_str.encode()).hexdigest()
        if seq_hash == hist_hash:
            return True
    return False

# History from train-normal windows
history_seqs = X_train[: min(5000, len(X_train))]

replay_flags = [is_replay_attack(seq, history_seqs) for seq in X_eval]

# final multiclass status
final_status = []
for i, pred in enumerate(y_pred_binary):
    if replay_flags[i] and pred == 1:
        final_status.append('Data Manipulation (Replay Attack)')
    elif pred == 1:
        final_status.append('Environmental Anomaly')
    else:
        final_status.append('Normal')


# 7. Thorough Evaluation (No Overfitting / Repetition Issues)

In [ ]:
# Ground truth mapped to evaluation classes
true_labels = []
for lbl in seq_labels:
    s = str(lbl).lower()
    if 'replay' in s:
        true_labels.append('Data Manipulation (Replay Attack)')
    elif s in {'anomaly', 'attack', 'intrusion'}:
        true_labels.append('Environmental Anomaly')
    else:
        true_labels.append('Normal')

print('=== Classification Report ===')
print(classification_report(true_labels, final_status, zero_division=0))

# Confusion matrix (3 classes)
ordered_labels = ['Normal', 'Environmental Anomaly', 'Data Manipulation (Replay Attack)']
cm = confusion_matrix(true_labels, final_status, labels=ordered_labels)
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=ordered_labels,
    yticklabels=ordered_labels
)
plt.title('Confusion Matrix')
plt.show()

# Precision-Recall and ROC-AUC on binary anomaly detection
precision, recall, _ = precision_recall_curve(y_eval_binary, eval_mse)
plt.plot(recall, precision)
plt.title('Precision-Recall Curve (Binary)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.show()

if len(np.unique(y_eval_binary)) < 2:
    print('⚠️ ROC-AUC not defined (single class in y_eval_binary).')
else:
    auc = roc_auc_score(y_eval_binary, eval_mse)
    print(f'AUC-ROC (anomaly detection): {auc:.4f}')


# 8. Inference Pipeline (Real-time Ready)

In [ ]:
def detect_anomaly(new_sequence, model, threshold, history_seqs):
    new_seq = np.array(new_sequence).reshape(1, 60, 2)
    recon = model.predict(new_seq, verbose=0)
    mse = np.mean(np.power(new_seq - recon, 2))

    if mse > threshold:
        if is_replay_attack(new_seq[0], history_seqs):
            return '🚨 DATA MANIPULATION ATTACK (Replay)'
        return '🚨 ENVIRONMENTAL ANOMALY'
    return '✅ Normal'

# Example inference
example_window = df[['temperature', 'humidity']].iloc[100:160].values
print(detect_anomaly(example_window, model, threshold, history_seqs))
